In [ ]:
from scdesigner.datasets import pancreas

exper = pancreas()
exper

### Composite Definition

Here is an example of how we can can fit different formulas and models across different subsets of genes, while keeping them all in the same simulator object. This could be accomplished through several calls to `.fit()`, but the `Composite` interaface gives a more convenient shorthand.

In [ ]:
from scdesigner.simulators import CompositeCopula, NegBinCopula, NegBinCopula

specification = {
    "group1": {"formula": "~ pseudotime", "simulator": NegBinCopula(epochs=10), "var_names": exper.var_names[:50]},
    "group2": {"formula": "~ 1", "simulator": NegBinCopula(epochs=10), "var_names": exper.var_names[50:]}
}

sim = CompositeCopula(specification)
sim.fit(exper)
sim

Now that the simulators are tied together in this way, we can get local parameter predictions across all groups through a single `predict` call.

In [ ]:
sim.predict(exper.obs)

Simulated `AnnData` objects are automatically concatenated across groups.

In [ ]:
sim.sample(exper.obs)

### Splitting a Simulator

One goal of `scDesigner` is to provide a grammar for manipulating already-trained simulators, so that we can have fine-grained of synthetic null/alternative generation without requiring re-estimating the model from scratch. We already have `transform` functions `nullify` and `amplify` for modifying parameters through string-matching. But we may want to re-estimate a part of a model, switch the input variables, or change the distribution. In this case, we can split a single simulator into a composite of several submodels. Note that we only re-fit the submodels that don't yet have parameters fitted. To illustrate, we first fit an ordinary simulator across all genes.

In [ ]:
from scdesigner.transform import split_glm

sim = NegBinCopula(epochs=10)
sim.fit(exper, "~ pseudotime")
sim.params["coef_mean"]

Let's now refit the first 10 genes without pseudotime as a predictor. This is related to nullifying those genes, except we also re-estimate the intercept terms. This is important in the case that the nullified variable is correlated to the other terms in the regression formula. After refitting, we're left with a composite (not NB) simulator. By default, the new "split" is given the key "group2", but this can be modified in the `split_glm` arguments.

In [ ]:
sim_split = split_glm(sim, {"var_names": exper.var_names[:10], "formula": "~ 1"})
sim_split.fit(exper)
sim_split.params["group2"]["coef_mean"]

Let's double check that the ten refitted genes have been removed entirely from the initial model.

In [ ]:
sim_split.params["group1"]["covariance"].shape

Nonetheless, when we sample the composite simulator, it internally combines sampled output across the genes.

In [ ]:
sim_split.sample(exper.obs)

By default, it uses the same simulator type as we initially trained on. We can alternatively keep the formula the same but modify the model.

In [ ]:
sim_split = split_glm(sim, {"var_names": exper.var_names[:10], "simulator": NegBinCopula(epochs=4)})
sim_split.fit(exper)
sim_split.params["group2"]["coef_mean"]

We can also modify both the formula and the model simultaneously.

In [ ]:
sim_split = split_glm(sim, {"var_names": exper.var_names[:10], "simulator": NegBinCopula(epochs=4)})
sim_split.fit(exper)
sim_split.params["group2"]["coef_mean"]